# 聚类算法（Clustering）

聚类是一种**无监督学习**方法：训练数据没有标签，算法自动发现数据内部的分组结构。

**与分类的区别**：
- 分类（有监督）：已知类别，学习如何把新样本归入已知类
- 聚类（无监督）：不知道类别，让算法自己找出「相似样本」的自然分组

常见聚类算法：
| 算法 | 核心思想 | 适用场景 |
|------|----------|----------|
| **K-Means** | 以距离为相似度，迭代更新簇中心 | 球形簇、数量已知 |
| **层次聚类** | 逐步合并/分裂，形成树状结构 | 需要查看层级关系 |
| **DBSCAN** | 以密度定义簇，自动发现任意形状 | 非球形簇、含噪声点 |

## 1. K-Means 算法

### 1.1 算法原理

目标：将 $n$ 个样本划分为 $K$ 个簇，使每个样本到所属簇中心的距离平方和最小：

$$J = \sum_{k=1}^{K} \sum_{x_i \in C_k} \|x_i - \mu_k\|^2$$

**迭代步骤：**
1. 随机初始化 $K$ 个簇中心 $\mu_1, \ldots, \mu_K$
2. **分配**：将每个样本分配到距离最近的簇中心
3. **更新**：重新计算每个簇的中心（均值）
4. 重复 2-3 直到簇分配不再变化（收敛）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.preprocessing import StandardScaler

# 生成三簇数据
X, y_true = make_blobs(n_samples=300, centers=3, cluster_std=0.8, random_state=42)

# 手动实现 K-Means 并记录每步过程
np.random.seed(42)
K = 3
centers = X[np.random.choice(len(X), K, replace=False)]  # 随机初始化

history = [centers.copy()]
for _ in range(10):
    # 分配步骤
    dists   = np.linalg.norm(X[:, None] - centers[None], axis=2)  # (n, K)
    labels  = np.argmin(dists, axis=1)
    # 更新步骤
    new_centers = np.array([X[labels == k].mean(axis=0) for k in range(K)])
    history.append(new_centers.copy())
    if np.allclose(centers, new_centers):
        break
    centers = new_centers

# 可视化收敛过程（取第1、2、最终 步）
steps = [0, 1, len(history) - 1]
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
colors = ['steelblue', 'tomato', 'seagreen']

for ax, step in zip(axes, steps):
    c = history[step]
    dists  = np.linalg.norm(X[:, None] - c[None], axis=2)
    lbl    = np.argmin(dists, axis=1)
    for k in range(K):
        ax.scatter(X[lbl==k, 0], X[lbl==k, 1], c=colors[k], s=15, alpha=0.6)
    ax.scatter(c[:, 0], c[:, 1], c='black', marker='X', s=200, zorder=5, label='簇中心')
    ax.set_title(f'迭代第 {step} 步')
    ax.legend(fontsize=8)

plt.suptitle('K-Means 收敛过程', fontsize=13)
plt.tight_layout()
plt.show()

### 1.2 肘部法则（Elbow Method）选择 K 值

K-Means 需要预先指定 $K$。**肘部法则**：
- 对不同的 $K$ 计算惯性（Inertia）= 所有样本到所属簇中心的距离平方和
- Inertia 随 $K$ 增大单调递减
- **拐点（肘部）**处是最合适的 $K$：继续增大 $K$ 收益递减

In [ ]:
from sklearn.cluster import KMeans

inertias = []
K_range = range(1, 10)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(K_range, inertias, marker='o', color='steelblue', linewidth=2)
plt.axvline(3, color='red', linestyle='--', label='最优 K=3（肘部）')
plt.xlabel('K（簇数量）')
plt.ylabel('Inertia（簇内误差平方和）')
plt.title('肘部法则')
plt.legend()
plt.tight_layout()
plt.show()

### 1.3 sklearn K-Means 实现

In [ ]:
km = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = km.fit_predict(X)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# 真实标签 vs 聚类结果
for k in range(3):
    ax1.scatter(X[y_true==k, 0], X[y_true==k, 1], s=15, alpha=0.7, label=f'真实簇 {k}')
ax1.set_title('真实分布')
ax1.legend(fontsize=8)

for k in range(3):
    ax2.scatter(X[labels==k, 0], X[labels==k, 1], s=15, alpha=0.7, label=f'聚类簇 {k}')
ax2.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
            c='black', marker='X', s=200, zorder=5, label='簇中心')
ax2.set_title('K-Means 聚类结果')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()
print(f"Inertia（簇内误差平方和）= {km.inertia_:.2f}")
print(f"迭代次数 = {km.n_iter_}")

### 1.4 K-Means 的局限性

K-Means 假设簇是**凸形（球状）**，面对非球形数据效果差。

In [ ]:
X_moon, _   = make_moons(n_samples=300, noise=0.05, random_state=42)
X_circle, _ = make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, Xd, title in zip(axes, [X_moon, X_circle], ['月牙形数据', '同心圆数据']):
    lbl = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(Xd)
    for k in range(2):
        ax.scatter(Xd[lbl==k, 0], Xd[lbl==k, 1], s=15, alpha=0.8)
    ax.set_title(f'K-Means 在{title}上的失败')

plt.tight_layout()
plt.show()
print("→ K-Means 对非球形簇束手无策，需要用 DBSCAN 等密度算法")

## 2. 层次聚类（Hierarchical Clustering）

### 2.1 算法原理

**凝聚型（自底向上）**：
1. 初始时每个样本自成一个簇
2. 每步将**距离最近**的两个簇合并
3. 重复直到所有样本合并为一个簇

**簇间距离的衡量方式（链接方法）：**
| 方法 | 定义 | 特点 |
|------|------|------|
| 单链接 (single) | 两簇中最近点的距离 | 容易形成长链 |
| 全链接 (complete) | 两簇中最远点的距离 | 倾向紧凑球形簇 |
| 均值链接 (average) | 所有跨簇点对距离均值 | 折中效果 |
| Ward 法 | 合并后使簇内方差增量最小 | 最常用，效果好 |

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering

# 用较少样本展示树状图
X_small, _ = make_blobs(n_samples=30, centers=3, random_state=42)

Z = linkage(X_small, method='ward')

plt.figure(figsize=(12, 4))
dendrogram(Z, leaf_rotation=90, leaf_font_size=8,
           color_threshold=Z[-3, 2])   # 在第3高处截断，对应3个簇
plt.axhline(Z[-3, 2], color='red', linestyle='--', label=f'截断线 → 3 个簇')
plt.xlabel('样本索引')
plt.ylabel('合并距离')
plt.title('层次聚类树状图（Ward 链接）')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 对比四种链接方法
methods = ['ward', 'complete', 'average', 'single']
fig, axes = plt.subplots(1, 4, figsize=(15, 4))

for ax, method in zip(axes, methods):
    agg = AgglomerativeClustering(n_clusters=3, linkage=method)
    lbl = agg.fit_predict(X)
    for k in range(3):
        ax.scatter(X[lbl==k, 0], X[lbl==k, 1], s=10, alpha=0.7)
    ax.set_title(f'linkage={method}')

plt.suptitle('层次聚类 - 四种链接方法对比', fontsize=13)
plt.tight_layout()
plt.show()

## 3. DBSCAN（密度聚类）

### 3.1 算法原理

DBSCAN（Density-Based Spatial Clustering of Applications with Noise）基于**密度**定义簇，不需要指定簇数量，能自动识别噪声点。

**两个关键参数：**
- `eps`（$\varepsilon$）：邻域半径，定义「附近」的范围
- `min_samples`：核心点的最少邻居数

**点的三种类型：**
- **核心点**：$\varepsilon$ 邻域内至少有 `min_samples` 个点
- **边界点**：在某核心点的邻域内，但自身不是核心点
- **噪声点**：既不是核心点也不是边界点（标记为 -1）

**簇的形成**：从核心点出发，将所有密度可达的点连成一个簇。

In [ ]:
from sklearn.cluster import DBSCAN

# DBSCAN 对月牙形和同心圆效果
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

params = [
    (X_moon,   {'eps': 0.2,  'min_samples': 5}, '月牙形'),
    (X_circle, {'eps': 0.15, 'min_samples': 5}, '同心圆'),
]

for ax, (Xd, kw, title) in zip(axes, params):
    db  = DBSCAN(**kw)
    lbl = db.fit_predict(Xd)
    unique_labels = set(lbl)
    palette = plt.cm.tab10(np.linspace(0, 1, len(unique_labels)))
    for color, k in zip(palette, sorted(unique_labels)):
        mask = lbl == k
        label = f'簇 {k}' if k >= 0 else '噪声'
        ax.scatter(Xd[mask, 0], Xd[mask, 1], c=[color], s=15,
                   alpha=0.8, label=label)
    ax.set_title(f'DBSCAN 在{title}上（eps={kw["eps"]}）')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# DBSCAN 的噪声点检测能力
np.random.seed(0)
X_noise = np.vstack([
    make_blobs(n_samples=200, centers=2, cluster_std=0.4, random_state=0)[0],
    np.random.uniform(-4, 4, size=(30, 2))  # 随机噪声点
])

db   = DBSCAN(eps=0.5, min_samples=5)
lbl  = db.fit_predict(X_noise)
n_clusters = len(set(lbl)) - (1 if -1 in lbl else 0)
n_noise    = np.sum(lbl == -1)

plt.figure(figsize=(7, 5))
mask_noise = lbl == -1
plt.scatter(X_noise[~mask_noise, 0], X_noise[~mask_noise, 1],
            c=lbl[~mask_noise], cmap='tab10', s=20, alpha=0.8, label='聚类点')
plt.scatter(X_noise[mask_noise, 0], X_noise[mask_noise, 1],
            c='black', marker='x', s=60, label=f'噪声点 ({n_noise}个)')
plt.title(f'DBSCAN 噪声检测（发现 {n_clusters} 个簇，{n_noise} 个噪声点）')
plt.legend()
plt.tight_layout()
plt.show()

### 3.2 eps 参数选择：K 距离图

用 **K 距离图**辅助选择 `eps`：计算每个点到第 `min_samples` 近邻的距离，排序后画图，**曲线拐点**即为合适的 `eps`。

In [ ]:
from sklearn.neighbors import NearestNeighbors

min_samples = 5
nbrs = NearestNeighbors(n_neighbors=min_samples).fit(X_noise)
distances, _ = nbrs.kneighbors(X_noise)
k_dist = np.sort(distances[:, -1])[::-1]  # 第 min_samples 近邻距离，降序排列

plt.figure(figsize=(8, 4))
plt.plot(k_dist, color='steelblue', linewidth=2)
plt.axhline(0.5, color='red', linestyle='--', label='建议 eps ≈ 0.5（拐点）')
plt.xlabel('样本（按距离降序排列）')
plt.ylabel(f'第 {min_samples} 近邻距离')
plt.title('K 距离图（辅助选择 eps）')
plt.legend()
plt.tight_layout()
plt.show()

## 4. 聚类评估指标

聚类是无监督学习，没有真实标签时很难客观评估。常用指标：

### 4.1 有真实标签时（外部指标）
- **ARI（调整兰德指数）**：$[-1, 1]$，越接近 1 越好，0 表示随机
- **NMI（标准化互信息）**：$[0, 1]$，越大越好

### 4.2 无真实标签时（内部指标）
- **轮廓系数（Silhouette Score）**：$[-1, 1]$，越接近 1 越好

$$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$$

$a(i)$：样本 $i$ 到**同簇**其他点的平均距离（簇内凝聚度）

$b(i)$：样本 $i$ 到**最近异簇**所有点的平均距离（簇间分离度）

- **CH 指数（Calinski-Harabasz）**：簇间散度 / 簇内散度，值越大越好

In [ ]:
from sklearn.metrics import (
    silhouette_score, silhouette_samples,
    adjusted_rand_score, normalized_mutual_info_score,
    calinski_harabasz_score
)

km_labels = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X)

print("=== 有真实标签（外部指标）===")
print(f"ARI  (调整兰德指数)  = {adjusted_rand_score(y_true, km_labels):.4f}")
print(f"NMI  (标准化互信息)  = {normalized_mutual_info_score(y_true, km_labels):.4f}")
print()
print("=== 无真实标签（内部指标）===")
print(f"轮廓系数 Silhouette  = {silhouette_score(X, km_labels):.4f}")
print(f"CH 指数              = {calinski_harabasz_score(X, km_labels):.2f}")

In [ ]:
# 用轮廓系数图辅助验证最优 K
from matplotlib import cm

K_list = [2, 3, 4, 5]
fig, axes = plt.subplots(1, len(K_list), figsize=(15, 4))

for ax, k in zip(axes, K_list):
    km_k   = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_k = km_k.fit_predict(X)
    sil_avg  = silhouette_score(X, labels_k)
    sil_vals = silhouette_samples(X, labels_k)

    y_lower = 10
    cmap = cm.get_cmap('tab10')
    for i in range(k):
        vals_i = np.sort(sil_vals[labels_k == i])
        size_i = vals_i.shape[0]
        y_upper = y_lower + size_i
        ax.fill_betweenx(np.arange(y_lower, y_upper), 0, vals_i,
                         alpha=0.7, color=cmap(i / k))
        y_lower = y_upper + 10

    ax.axvline(sil_avg, color='red', linestyle='--', linewidth=1.5)
    ax.set_title(f'K={k}  avg={sil_avg:.3f}')
    ax.set_xlabel('轮廓系数')
    ax.set_yticks([])

plt.suptitle('不同 K 值的轮廓系数图（红线=均值，越宽越好）', fontsize=12)
plt.tight_layout()
plt.show()

## 5. 三种算法综合对比

In [ ]:
from sklearn.cluster import AgglomerativeClustering

datasets = [
    ('球形簇',   *make_blobs(n_samples=300, centers=3, cluster_std=0.8, random_state=42)),
    ('月牙形',   *make_moons(n_samples=300, noise=0.05, random_state=42)),
    ('同心圆',   *make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=42)),
]

algorithms = [
    ('K-Means',      lambda X: KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(X)),
    ('层次聚类(Ward)', lambda X: AgglomerativeClustering(n_clusters=2, linkage='ward').fit_predict(X)),
    ('DBSCAN',       lambda X: DBSCAN(eps=0.3, min_samples=5).fit_predict(X)),
]

fig, axes = plt.subplots(len(datasets), len(algorithms), figsize=(13, 10))

for row, (name, Xd, _) in enumerate(datasets):
    Xd_s = StandardScaler().fit_transform(Xd)
    for col, (alg_name, alg_fn) in enumerate(algorithms):
        ax = axes[row][col]
        lbl = alg_fn(Xd_s)
        unique = set(lbl)
        for k in sorted(unique):
            mask = lbl == k
            label = f'噪声' if k == -1 else f'簇{k}'
            ax.scatter(Xd_s[mask, 0], Xd_s[mask, 1], s=10, alpha=0.7, label=label)
        sil = silhouette_score(Xd_s, lbl) if len(unique) > 1 and -1 not in unique else float('nan')
        ax.set_title(f'{name} / {alg_name}\nsil={sil:.3f}' if not np.isnan(sil) else f'{name} / {alg_name}')
        ax.set_xticks([])
        ax.set_yticks([])

plt.suptitle('三种聚类算法 × 三种数据集对比', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 总结

| 对比维度 | K-Means | 层次聚类 | DBSCAN |
|----------|---------|----------|--------|
| 需要指定 K | 是 | 是（或截断高度） | 否 |
| 簇形状假设 | 球形 | 取决于链接方法 | 任意形状 |
| 噪声点处理 | 无 | 无 | 自动标记为 -1 |
| 时间复杂度 | $O(nKt)$ | $O(n^2)$ | $O(n \log n)$ |
| 大数据适用 | 好 | 差 | 中等 |
| 主要超参数 | `n_clusters` | `n_clusters`, `linkage` | `eps`, `min_samples` |

**选型建议：**
- 数据量大、簇大致为球形 → **K-Means**
- 需要了解层级关系、数据量小 → **层次聚类**
- 形状不规则、存在噪声、不知道 K → **DBSCAN**